<a href="https://colab.research.google.com/github/OlorteguiKevin/lab03/blob/develop/GLab_03_Miner%C3%ADa_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEMANA 3: ARQUITECTURA DE UN DATA WAREHOUSE. CARGA Y MANTENIMIENTO. CUBOS DE DATOS

## Alumno: Olortegui Perez Kevin Estiben

In [53]:
# Importamos las librerías necesarias
import pandas as pd

In [54]:
# Definimos la URL del dataset
# Se asigna la URL del dataset "Car Evaluation" del UCI Machine Learning Repository.
# Este enlace debe estar concatenado con el nombre del archivo de datos.
# Link :https://archive.ics.uci.edu/ml/datasets/Car+Evaluation
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

In [55]:
# Cargamos los datos en un DataFrame
# Se usa pd.read_csv() para leer el archivo CSV desde la URL. En caso de que no se
# reconozcan los datos correctamente, podemos verificar si el separador es diferente o si los nombres de los atributos deben ajustarse.
data = pd.read_csv(url)
data

,vhigh,vhigh.1,2,2.1,small,low,unacc
0,vhigh,vhigh,2,2,small,med,unacc
1,vhigh,vhigh,2,2,small,high,unacc
2,vhigh,vhigh,2,2,med,low,unacc
3,vhigh,vhigh,2,2,med,med,unacc
4,vhigh,vhigh,2,2,med,high,unacc
...,...,...,...,...,...,...,...
1722,low,low,5more,more,med,med,good
1723,low,low,5more,more,med,high,vgood
1724,low,low,5more,more,big,low,unacc
1725,low,low,5more,more,big,med,good


In [56]:
# Asignamos nombres a las columnas
# El dataset original no tiene nombres de columnas, por lo que los asignamos manualmente.
data = pd.read_csv(url,names=["buying", "maint", "doors", "persons",
"lug_boot", "safety", "acceptability"])
data

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [57]:
# Extraer columnas
list(data)
list(data.columns)
data.columns.tolist()
list(data.columns.values)


['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'acceptability']

In [58]:
# Medición del rendimiento
from timeit import timeit
t1 = timeit(lambda: list(data))
t2 = timeit(lambda: list(data.columns))
t3 = timeit(lambda: data.columns.tolist())
t4 = timeit(lambda: list(data.columns.values))
print(t1,t2,t3,t4)

2.1391086789999463 1.4531321589997788 0.6621786149999025 2.330442103999758


In [59]:
data['buying'].dtype
data['persons' ].dtype
data.dtypes

,0
buying,object
maint,object
doors,object
persons,object
lug_boot,object
safety,object
acceptability,object


In [60]:
data["doors"].unique()

array(['2', '3', '4', '5more'], dtype=object)

In [61]:
# Cambiando tipo de datos
data['doors'] = data['doors']. astype('category')
data['persons'] = data['persons'].astype('category')
data.dtypes


,0
buying,object
maint,object
doors,category
persons,category
lug_boot,object
safety,object
acceptability,object


In [62]:
# Cambiando nombre a categorías
import numpy as np
data["lug_boot"] = np.where(data["lug_boot"] == "small", "pequeño", data["lug_boot"])
data["lug_boot"] = np.where(data["lug_boot"] == "med", "mediano", data["lug_boot"])
data["lug_boot"] = np.where(data["lug_boot"] == "big", "grande", data["lug_boot"])
data["lug_boot"]. unique()

array(['pequeño', 'mediano', 'grande'], dtype=object)

In [63]:
# Con el método '.value_count( )' podemos contabilizar las frecuencias de cada categoría.
data['doors' ].value_counts(dropna=False)

,count
doors,
2,432
3,432
4,432
5more,432


In [64]:
# También podemos juntar dos o más categorías mediante el método 'where' escribiendo la misma
# categoría para las que son diferentes.
data["doors"] = np.where(data["doors"] == "2", "3 a menos", data["doors"])
data["doors"] = np.where(data["doors"] == "3", "3 a menos", data["doors"])
data["doors"] = np.where(data["doors"] == "4", "4 a más", data["doors"])
data["doors"] = np.where(data["doors"] == "5more", "4 a más", data["doors"])
data["doors"]. unique()

array(['3 a menos', '4 a más'], dtype=object)

In [65]:
# Usamos "rename" si queremos cambiar los encabezados de otra manera
data.rename({'lug_boot': 'trunk'}, axis=1, inplace=True)
data


,buying,maint,doors,persons,trunk,safety,acceptability
0,vhigh,vhigh,3 a menos,2,pequeño,low,unacc
1,vhigh,vhigh,3 a menos,2,pequeño,med,unacc
2,vhigh,vhigh,3 a menos,2,pequeño,high,unacc
3,vhigh,vhigh,3 a menos,2,mediano,low,unacc
4,vhigh,vhigh,3 a menos,2,mediano,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,4 a más,more,mediano,med,good
1724,low,low,4 a más,more,mediano,high,vgood
1725,low,low,4 a más,more,grande,low,unacc
1726,low,low,4 a más,more,grande,med,good


In [66]:
# Número de registros y atributos
data. shape
data.head()
data.tail()


,buying,maint,doors,persons,trunk,safety,acceptability
1723,low,low,4 a más,more,mediano,med,good
1724,low,low,4 a más,more,mediano,high,vgood
1725,low,low,4 a más,more,grande,low,unacc
1726,low,low,4 a más,more,grande,med,good
1727,low,low,4 a más,more,grande,high,vgood


In [67]:
# Segmentación de filas y columnas

data.iloc[0:3,1:4]

,maint,doors,persons
0,vhigh,3 a menos,2
1,vhigh,3 a menos,2
2,vhigh,3 a menos,2


In [68]:
# Extraemos registros en particular
data.loc[[0,10], : ]

,buying,maint,doors,persons,trunk,safety,acceptability
0,vhigh,vhigh,3 a menos,2,pequeño,low,unacc
10,vhigh,vhigh,3 a menos,4,pequeño,med,unacc


In [69]:
# Extraemos los datos de dos atributos
data.loc[0:20, ['doors', 'persons' ] ]

,doors,persons
0,3 a menos,2
1,3 a menos,2
2,3 a menos,2
3,3 a menos,2
4,3 a menos,2
5,3 a menos,2
6,3 a menos,2
7,3 a menos,2
8,3 a menos,2
9,3 a menos,4


In [70]:
# Extraemos los datos de una categoría en particular
data[data.buying == "vhigh"]

,buying,maint,doors,persons,trunk,safety,acceptability
0,vhigh,vhigh,3 a menos,2,pequeño,low,unacc
1,vhigh,vhigh,3 a menos,2,pequeño,med,unacc
2,vhigh,vhigh,3 a menos,2,pequeño,high,unacc
3,vhigh,vhigh,3 a menos,2,mediano,low,unacc
4,vhigh,vhigh,3 a menos,2,mediano,med,unacc
...,...,...,...,...,...,...,...
427,vhigh,low,4 a más,more,mediano,med,acc
428,vhigh,low,4 a más,more,mediano,high,acc
429,vhigh,low,4 a más,more,grande,low,unacc
430,vhigh,low,4 a más,more,grande,med,acc


In [71]:
# utilizamos operadores lógiocs para combinar condiciones
data[(data.safety == "med") | (data.safety == "high")]

,buying,maint,doors,persons,trunk,safety,acceptability
1,vhigh,vhigh,3 a menos,2,pequeño,med,unacc
2,vhigh,vhigh,3 a menos,2,pequeño,high,unacc
4,vhigh,vhigh,3 a menos,2,mediano,med,unacc
5,vhigh,vhigh,3 a menos,2,mediano,high,unacc
7,vhigh,vhigh,3 a menos,2,grande,med,unacc
...,...,...,...,...,...,...,...
1721,low,low,4 a más,more,pequeño,high,good
1723,low,low,4 a más,more,mediano,med,good
1724,low,low,4 a más,more,mediano,high,vgood
1726,low,low,4 a más,more,grande,med,good


In [72]:
# Si deseamos exportar nuestro documento a un csv adicionándole la ruta dónde queremos guardar
# nuestro documento
ruta = 'F:\\Tecsup\\Cursos\\Minería de Datos\\data.csv'
data.to_csv(ruta)
